# Analyse de survie des intervalles intergénésiques au Cameroun (EDS 2018)

## Partie Machine Learning

Objectif : prédire les intervalles intergénésiques courts (<24 mois).

Workflow complet :
1. Importation
2. Chargement du dataset
3. EDA
4. Prétraitement
5. SMOTE
6. Entraînement de plusieurs modèles
7. Évaluation
8. ROC-AUC
9. Matrices de confusion
10. SHAP
11. Sauvegarde du meilleur modèle


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
import shap


## Chargement des données

In [2]:
df = pd.read_csv('df_ml_intergenesique.csv')
df.head()

,temps_mois,evenement_naissance,poids_sondage,milieu_residence,niveau_instruction,indice_richesse,region,religion,sexe_enfant,survie_enfant_precedent,age_premiere_naissance,instruction_mari
0,29,1,0.643138,Rural,Secondaire,Pauvre,Sud,Catholique,Masculin,Vivant,16-20 ans,NaN
1,33,1,0.643138,Rural,Secondaire,Pauvre,Sud,Catholique,Masculin,Vivant,16-20 ans,NaN
2,24,1,0.643138,Rural,Secondaire,Pauvre,Sud,Catholique,Masculin,Vivant,16-20 ans,NaN
3,38,1,0.643138,Rural,Secondaire,Pauvre,Sud,Catholique,Masculin,Vivant,16-20 ans,NaN
4,23,0,0.643138,Rural,Secondaire,Pauvre,Sud,Catholique,Masculin,Vivant,16-20 ans,NaN


## Construction de la variable cible

In [3]:
SEUIL=24

df['Y']=(df['temps_mois']<SEUIL).astype(int)


## Prétraitement

In [4]:
X=df.drop(columns=['temps_mois','evenement_naissance','poids_sondage','Y'])
y=df['Y']

num_features=['age_premiere_naissance']
cat_features=[c for c in X.columns if c not in num_features]

preprocessor=ColumnTransformer([
('num',Pipeline([
('imputer',SimpleImputer(strategy='median')),
('scaler',StandardScaler())]),num_features),

('cat',Pipeline([
('imputer',SimpleImputer(strategy='most_frequent')),
('encoder',OneHotEncoder(handle_unknown='ignore'))]),cat_features)
])


## Split + SMOTE

In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

X_train_prep=preprocessor.fit_transform(X_train)
X_test_prep=preprocessor.transform(X_test)

smote=SMOTE(random_state=42)
X_train_smote,y_train_smote=smote.fit_resample(X_train_prep,y_train)


ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: '21-25 ans'

## Modèles

In [ ]:
models={
'LR':LogisticRegression(max_iter=1000),
'RF':RandomForestClassifier(),
'ExtraTrees':ExtraTreesClassifier(),
'SVM':SVC(probability=True),
'KNN':KNeighborsClassifier(),
'XGB':xgb.XGBClassifier(),
'LGBM':lgb.LGBMClassifier(verbose=-1)
}


## Entraînement et comparaison

In [ ]:
results=[]
probas={}

for name,model in models.items():
    model.fit(X_train_smote,y_train_smote)
    pred=model.predict(X_test_prep)
    p=model.predict_proba(X_test_prep)[:,1]
    auc=roc_auc_score(y_test,p)
    probas[name]=p
    results.append([name,auc])

results=pd.DataFrame(results,columns=['Modele','AUC'])
results.sort_values('AUC',ascending=False)


## Courbes ROC

In [ ]:
plt.figure(figsize=(8,6))

for name,p in probas.items():
    fpr,tpr,_=roc_curve(y_test,p)
    plt.plot(fpr,tpr,label=name)

plt.legend()
plt.show()


## SHAP

In [ ]:
best_name=results.sort_values('AUC',ascending=False).iloc[0,0]
best_model=models[best_name]

explainer=shap.Explainer(best_model)
shap_values=explainer(X_test_prep)
shap.plots.beeswarm(shap_values)
